# 02 - Data Cleaning

##  Objectif
Nettoyer les données brutes pour préparer le **Feature Engineering**

##  Stratégie
Basé sur l'EDA, nous allons :
1. **Parser les colonnes complexes** (nutrition, tags, ingredients, steps)
2. **Nettoyer les valeurs aberrantes** (outliers, missing values)
3. **Filtrer les données invalides** (duplicatas, valeurs impossibles)
4. **Exporter les fichiers cleaned** prêts pour le feature engineering

## Outputs :
- **`recipes_clean.csv`** : Recettes nettoyées avec nutrition + tags parsés
- **`interactions_clean.csv`** : Interactions filtrées (ratings valides)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style("whitegrid")
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

# Helper pour parser les colonnes liste
def parse_list_col(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x if isinstance(x, list) else []

print('✅ Configuration OK')
print(f'  Data raw: {DATA_RAW}')
print(f'  Data processed: {DATA_PROCESSED}')


In [ ]:
print("=" * 80)
print("CHARGEMENT DES DONNÉES")
print("=" * 80)

# Charger les 3 fichiers principaux
interactions = pd.read_csv(DATA_RAW / 'interactions_train.csv')
recipes = pd.read_csv(DATA_RAW / 'RAW_recipes.csv')
users = pd.read_csv(DATA_RAW / 'PP_users.csv')

print(f'\n DONNÉES CHARGÉES:')
print(f'   • Interactions: {interactions.shape}')
print(f'   • Recipes:      {recipes.shape}')
print(f'   • Recipes list:      {recipes.columns.tolist()}')
print(f'   • Users:        {users.shape}')

print(f'\n UTILISATEURS UNIQUES:')
print(f'   • Dans interactions: {interactions["user_id"].nunique():,}')
print(f'   • Dans users.csv:    {users["u"].nunique():,}')

print(f'\n RECETTES UNIQUES:')
print(f'   • Dans interactions: {interactions["recipe_id"].nunique():,}')
print(f'   • Dans recipes.csv:  {recipes["id"].nunique():,}')


In [ ]:
print("=" * 80)
print("ÉTAPE 1 - PARSER NUTRITION ET TAGS DES RECETTES")
print("=" * 80)

# 1. Parser la colonne nutrition (7 valeurs)
recipes['nutrition_parsed'] = recipes['nutrition'].apply(parse_list_col)
# 2. Parser les tags et créer 7 catégories binaires (basées sur EDA)
recipes['tags_parsed'] = recipes['tags'].apply(parse_list_col)
recipes['steps_parsed'] = recipes['steps'].apply(parse_list_col)
recipes['ingredients_parsed'] = recipes['ingredients'].apply(parse_list_col)

recipes['calories'] = recipes['nutrition_parsed'].apply(lambda x: x[0] if len(x) > 0 else np.nan)
recipes['total_fat'] = recipes['nutrition_parsed'].apply(lambda x: x[1] if len(x) > 1 else np.nan)
recipes['sugar'] = recipes['nutrition_parsed'].apply(lambda x: x[2] if len(x) > 2 else np.nan)
recipes['sodium'] = recipes['nutrition_parsed'].apply(lambda x: x[3] if len(x) > 3 else np.nan)
recipes['protein'] = recipes['nutrition_parsed'].apply(lambda x: x[4] if len(x) > 4 else np.nan)
recipes['saturated_fat'] = recipes['nutrition_parsed'].apply(lambda x: x[5] if len(x) > 5 else np.nan)
recipes['carbohydrates'] = recipes['nutrition_parsed'].apply(lambda x: x[6] if len(x) > 6 else np.nan)

nutrition_cols = ['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates']


In [ ]:
print(recipes['tags_parsed'].head().tolist())
print(recipes['steps_parsed'].head().tolist())
print(recipes['ingredients_parsed'].head().tolist())

In [ ]:
print("=" * 80)
print("EDA - TAGS & INGREDIENTS")
print("=" * 80)

# Analyse TAGS
recipes['tags_parsed'] = recipes['tags'].apply(parse_list_col)
recipes['n_tags'] = recipes['tags_parsed'].apply(len)

all_tags = []
for tag_list in recipes['tags_parsed']:
    all_tags.extend(tag_list)
tag_counts = Counter(all_tags)

print(f"\n  TAGS:")
print(f"   • Total tags uniques: {len(tag_counts):,}")
print(f"   • Total tags (avec répétitions): {len(all_tags):,}")
print(f"   • Tags par recette (moyenne): {recipes['n_tags'].mean():.1f}")
print(f"   • Tags par recette (médiane): {recipes['n_tags'].median():.0f}")

# Analyse INGREDIENTS
recipes['n_ingredients_check'] = recipes['ingredients_parsed'].apply(len)

all_ingredients = []
for ing_list in recipes['ingredients_parsed']:
    all_ingredients.extend(ing_list)
ingredient_counts = Counter(all_ingredients)

print(f"\n INGREDIENTS:")
print(f"   • Total ingredients uniques: {len(ingredient_counts):,}")
print(f"   • Total ingredients (avec répétitions): {len(all_ingredients):,}")
print(f"   • Ingredients par recette (moyenne): {recipes['n_ingredients_check'].mean():.1f}")
print(f"   • Ingredients par recette (médiane): {recipes['n_ingredients_check'].median():.0f}")

# Analyse Steps
recipes['steps_parsed'] = recipes['steps'].apply(parse_list_col)
recipes['n_steps'] = recipes['steps_parsed'].apply(len)

all_steps = []
for step_list in recipes['steps_parsed']:
    all_steps.extend(step_list)
step_counts = Counter(all_steps)

print(f"\n ETAPES:")
print(f"   • Total étapes uniques: {len(step_counts):,}")
print(f"   • Total étapes (avec répétitions): {len(all_steps):,}")
print(f"   • Étapes par recette (moyenne): {recipes['n_steps'].mean():.1f}")
print(f"   • Étapes par recette (médiane): {recipes['n_steps'].median():.0f}")

print(f"\n TOP 30 TAGS:")
for tag, count in tag_counts.most_common(30):
    pct = 100 * count / len(recipes)
    print(f"   {tag:<40} {count:>7,} ({pct:>5.1f}%)")

print(f"\n TOP 30 INGREDIENTS:")
for ing, count in ingredient_counts.most_common(30):
    pct = 100 * count / len(recipes)
    print(f"   {ing:<40} {count:>7,} ({pct:>5.1f}%)")

print(f"\n TOP 30 ETAPES:")
for step, count in step_counts.most_common(30):
    pct = 100 * count / len(recipes)
    print(f"   {step:<40} {count:>7,} ({pct:>5.1f}%)")

## ÉTAPE 2 - Nettoyer les Recettes (Outliers & Missing Values)

In [ ]:
print("=" * 80)
print("NETTOYAGE DES RECETTES - OUTLIERS & MISSING VALUES")
print("=" * 80)

# 1. Imputer valeurs manquantes (médiane)
print(f'\n IMPUTATION (médiane):')
for col in nutrition_cols:
    n_missing = recipes[col].isnull().sum()
    if n_missing > 0:
        median = recipes[col].median()
        recipes[col].fillna(median, inplace=True)
        print(f'   • {col}: {n_missing:,} imputés → médiane={median:.1f}')

# 2. Winsorization (capper les outliers au 99e percentile)
print(f'\n  WINSORIZATION (99e/1e percentile):')
total_outliers = 0
for col in nutrition_cols + ['minutes']:
    p99, p1 = recipes[col].quantile([0.99, 0.01])
    n_above = (recipes[col] > p99).sum()
    n_below = (recipes[col] < p1).sum()
    total_outliers += n_above + n_below
    recipes.loc[recipes[col] > p99, col] = p99
    recipes.loc[recipes[col] < p1, col] = p1
    if n_above + n_below > 0:
        print(f'   • {col}: {n_above + n_below:,} cappés (p1={p1:.1f}, p99={p99:.1f})')

print(f'\n   Total outliers cappés: {total_outliers:,}')

# 3. Filtrer recettes invalides
print(f'\n FILTRAGE RECETTES INVALIDES:')
print(f'   Avant: {len(recipes):,}')

recipes_clean = recipes.copy()
recipes_clean = recipes_clean.dropna(subset=['id', 'name'])
recipes_clean = recipes_clean.drop_duplicates(subset=['id'])
recipes_clean = recipes_clean[(recipes_clean['minutes'] > 0) & (recipes_clean['minutes'] <= 1440)]
recipes_clean = recipes_clean[recipes_clean['calories'] > 0]
recipes_clean = recipes_clean[(recipes_clean['n_steps'] > 0) & (recipes_clean['n_ingredients'] > 0)]

print(f'   Après: {len(recipes_clean):,}')
print(f'   Supprimées: {len(recipes) - len(recipes_clean):,} ({100*(len(recipes)-len(recipes_clean))/len(recipes):.1f}%)')

print(f'\n✅ Recettes nettoyées: {recipes_clean.shape}')

## ÉTAPE 3 - Nettoyer les Interactions

In [ ]:
print("=" * 80)
print("NETTOYAGE DES INTERACTIONS")
print("=" * 80)

# Filtrer les interactions invalides
print(f'\n FILTRAGE INTERACTIONS INVALIDES:')
print(f'   Avant: {len(interactions):,}')

interactions_clean = interactions.copy()
# Garder seulement ratings valides (0-5)
interactions_clean = interactions_clean[interactions_clean['rating'].between(0, 5)]
# Supprimer duplicatas
interactions_clean = interactions_clean.drop_duplicates()
# Supprimer valeurs manquantes critiques
interactions_clean = interactions_clean.dropna(subset=['user_id', 'recipe_id', 'rating'])

print(f'   Après: {len(interactions_clean):,}')
print(f'   Supprimées: {len(interactions) - len(interactions_clean):,} ({100*(len(interactions)-len(interactions_clean))/len(interactions):.1f}%)')

print(f'\n✅ Interactions nettoyées: {interactions_clean.shape}')


## ÉTAPE 4 - Export des Fichiers Cleaned

In [ ]:
print("=" * 80)
print("EXPORT DES FICHIERS CLEANED")
print("=" * 80)

# Sélectionner les colonnes à exporter pour recipes
recipes_export_cols = [
    'id', 'name', 'minutes', 'n_steps', 'n_ingredients',
    'calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates',
    'tags_parsed', 'ingredients_parsed', 'steps_parsed'
]

# Export recipes_clean.csv
recipes_output = DATA_PROCESSED / 'recipes_clean.csv'
recipes_clean[recipes_export_cols].to_csv(recipes_output, index=False)

print(f'\n✅ FICHIER 1 - RECIPES_CLEAN.CSV:')
print(f'   • Chemin: {recipes_output}')
print(f'   • Shape: {recipes_clean[recipes_export_cols].shape}')
print(f'   • Taille: {recipes_output.stat().st_size / 1024 / 1024:.2f} MB')

print(f'   • Colonnes: {len(recipes_export_cols)}')

print(f'\n✅ Fichiers prêts pour Feature Engineering!')

# Export interactions_clean.csv

interactions_output = DATA_PROCESSED / 'interactions_clean.csv'
print(f'   • Colonnes: {len(interactions_clean.columns)}')

interactions_clean.to_csv(interactions_output, index=False)
print(f'   • Taille: {interactions_output.stat().st_size / 1024 / 1024:.2f} MB')

print(f'   • Shape: {interactions_clean.shape}')

print(f'\n✅ FICHIER 2 - INTERACTIONS_CLEAN.CSV:')
print(f'   • Chemin: {interactions_output}')

##  Synthèse Finale

In [ ]:
print("\n" + "=" * 80)
print(" SYNTHÈSE FINALE - DATA CLEANING")
print("=" * 80)

print(f'\n OBJECTIF ATTEINT:')
print(f'   Nettoyer et préparer les données pour le Feature Engineering')

print(f'\n DONNÉES SOURCES:')
print(f'   • Interactions (train):  {len(interactions):,}')
print(f'   • Recettes (RAW):        {len(recipes):,}')
print(f'   • Users (PP):            {len(users):,}')

print(f'\n TRAITEMENTS RÉALISÉS:')
print(f'   1. ✅ Parser nutrition (7 colonnes float)')
print(f'   2. ✅ Parser tags (7 catégories binaires)')
print(f'   3. ✅ Imputer valeurs manquantes (médiane)')
print(f'   4. ✅ Winsorizer outliers (99e/1e percentile)')
print(f'   5. ✅ Filtrer recettes invalides')
print(f'   6. ✅ Filtrer interactions invalides')

print(f'\n FICHIERS EXPORTÉS:')
print(f'   1. recipes_clean.csv')
print(f'      • Shape: {recipes_clean[recipes_export_cols].shape}')
print(f'      • Colonnes: {len(recipes_export_cols)} (nutrition + tags parsés)')
print(f'   2. interactions_clean.csv')
print(f'      • Shape: {interactions_clean.shape}')
print(f'      • Ratings valides: 0-5')

print(f'\n PROCHAINE ÉTAPE:')
print(f'   ✅ Notebook: 03_feature_engineering.ipynb')
print(f'      → Créer profils utilisateurs (agrégation par user_id)')
print(f'      → Export: users_profiles.csv')
print("=" * 80)
